# No-show data exploration (Kaggle "Medical Appointment No Shows")

Exploration only: nothing in `fullcut_ml` imports from this notebook. Run with `uv run --group notebook jupyter nbconvert --execute --inplace notebooks/01_explore.ipynb`.

In [1]:
import pandas as pd

from fullcut_ml.adapters.kaggle import load_kaggle_csv
from fullcut_ml.features import build_feature_frame
from fullcut_ml.train import DEFAULT_CSV

raw = pd.read_csv(DEFAULT_CSV)
raw.dtypes.to_frame("dtype")

,dtype
PatientId,float64
AppointmentID,int64
Gender,str
ScheduledDay,str
AppointmentDay,str
Age,int64
Neighbourhood,str
Scholarship,int64
Hipertension,int64
Diabetes,int64


## Columns: dataset vs what Fullcut will have

| Kaggle column | Fullcut equivalent | Used? |
|---|---|---|
| PatientId | `clients.id` | yes (history key) |
| AppointmentID | `bookings.id` | yes |
| ScheduledDay | `bookings.created_at` | yes (lead time) |
| AppointmentDay (date only) | `bookings.starts_at` (exact time) | yes; no hour in Kaggle |
| SMS_received | reminder sent (`scheduled_messages`) | yes |
| No-show | `bookings.status = no_show` | label |
| Gender, Age, Neighbourhood | not collected | **excluded** (fairness) |
| Scholarship, Hipertension, Diabetes, Alcoholism, Handcap | not collected | **excluded** (health/welfare data) |
| — | price, duration, source, staff, reply to confirm request, cancellations | Fullcut only |

The gap: this is Brazilian public-clinic data (free appointments, date-only, about 6 weeks of appointments). Salons charge, book to the minute and see the same client every 3-6 weeks, so client history should matter much more in Fullcut than it can here.

In [2]:
loaded = load_kaggle_csv(DEFAULT_CSV)
records = loaded.records
base_rate = sum(r.label for r in records) / len(records)
frame = build_feature_frame(records, shop_base_rate=base_rate)
frame["starts_at"] = pd.to_datetime(frame["starts_at"], utc=True)

print(f"rows read: {loaded.rows_read}, dropped (booked after appointment): {loaded.dropped_scheduled_after_appointment}")
print(f"bookings: {len(frame)}, clients: {frame.client_id.nunique()}")
print(f"appointments: {frame.starts_at.min().date()} to {frame.starts_at.max().date()}")
created = pd.Series([r.created_at for r in records])
print(f"booked: {created.min().date()} to {created.max().date()}")
print(f"no-show rate: {frame.label.mean():.3f}")

rows read: 110527, dropped (booked after appointment): 5
bookings: 110522, clients: 62299
appointments: 2016-04-29 to 2016-06-08
booked: 2015-11-10 to 2016-06-08
no-show rate: 0.202


In [3]:
frame["lead_bucket"] = pd.cut(
    frame.lead_time_days,
    [-0.1, 0.99, 3, 7, 14, 30, 60, 400],
    labels=["same day", "1-3 d", "4-7 d", "8-14 d", "15-30 d", "31-60 d", "60+ d"],
)
frame.groupby("lead_bucket", observed=True).label.agg(bookings="size", no_show_rate="mean")

,bookings,no_show_rate
lead_bucket,,
same day,43776,0.066361
1-3 d,9462,0.237371
4-7 d,17510,0.252027
8-14 d,12025,0.304699
15-30 d,17371,0.325888
31-60 d,8283,0.341543
60+ d,2095,0.284487


In [4]:
ax = frame.lead_time_days.clip(upper=90).plot.hist(bins=45, title="Lead time (days, clipped at 90)")
ax.set_xlabel("days between booking and appointment");

In [5]:
weekday_names = dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]))
by_day = frame.groupby("weekday", observed=True).label.agg(bookings="size", no_show_rate="mean")
by_day.index = [weekday_names[int(d)] for d in by_day.index]
by_day

,bookings,no_show_rate
Mon,22714,0.206437
Tue,25638,0.200874
Wed,25866,0.196861
Thu,17246,0.193494
Fri,19019,0.212261
Sat,39,0.230769


In [6]:
frame["history"] = pd.cut(
    frame.prior_bookings, [-1, 0, 2, 100], labels=["first visit", "1-2 prior", "3+ prior"]
)
frame["prior_no_show"] = frame.prior_no_shows > 0
frame.groupby(["history", "prior_no_show"], observed=True).label.agg(bookings="size", no_show_rate="mean")

bookings  no_show_rate
history     prior_no_show                        
first visit False             79643      0.217885
1-2 prior   False             16937      0.142705
            True               5410      0.221627
3+ prior    False              4029      0.102507
            True               4503      0.206973

In [7]:
frame.groupby("sms_reminder_sent").agg(
    bookings=("label", "size"), no_show_rate=("label", "mean"), median_lead_days=("lead_time_days", "median")
)

,bookings,no_show_rate,median_lead_days
sms_reminder_sent,,,
0.0,75040,0.166978,0.000000
1.0,35482,0.275745,13.683125


SMS reminders look *worse* at first glance, but they were only sent for bookings made days ahead, which are riskier anyway: lead time confounds the SMS flag. The model sees both, so it can separate them; the raw rate can't be read as "SMS causes no-shows".